In [1]:
import pandas as pd
import numpy as np
import glob as gb,os

In [7]:
countrys = ["br", "es","jp", "au", "mx", "us"]

years = [2018,2019,2020,2021,2022,2023,2024,2025]

df = pd.concat(
    (pd.read_csv(f) for f in gb.glob("./*csv")),
    ignore_index=True
)

df = df[df["cc"].isin(countrys)]
df["week_end"] = pd.to_datetime(df["week_end"])
df["year"] = df["week_end"].dt.year
df = df[df["year"].isin(years)]
df["pontos"] = 201 - df["rank"]


In [8]:
arquivos = sorted(gb.glob("./MGDplus/charts/regional/*.csv"))

dfs = []
for f in arquivos:
    d = pd.read_csv(f, sep="\t", thousands=",")   
    d["arquivo"] = os.path.basename(f)
    dfs.append(d)

other_df = pd.concat(dfs, ignore_index=True)

In [9]:
d = other_df.rename(columns={
    "market": "cc",
    "ID": "track_id",
    "Track": "title",
    "Artist": "artists",
    "Streams": "streams",
    "Date": "date",
    "Position": "rank_diario",
})

d = d[["cc", "date", "track_id", "title", "artists", "streams"]].copy()
d["date"] = pd.to_datetime(d["date"])
d["cc"] = d["cc"].str.lower()      # 'br' (confira se os 6 datasets usam o mesmo código)

# 1. Rótulo da semana = quinta-feira final (semana sex-qui, igual ao Spotify)
d["week_end"] = d["date"] + pd.to_timedelta((3 - d["date"].dt.weekday) % 7, unit="D")

# 2. Mantém só semanas completas (7 dias distintos) por país
dias_semana = d.groupby(["cc", "week_end"])["date"].transform("nunique")
d = d[dias_semana == 7]

# 3. Soma diária -> semanal por música
semanal = (
    d.groupby(["cc", "week_end", "track_id"], as_index=False)
     .agg(streams=("streams", "sum"),
          title=("title", "first"),
          artists=("artists", "first"))
)

# 4. Recria o rank semanal
semanal["rank"] = (
    semanal.groupby(["cc", "week_end"])["streams"]
           .rank(method="first", ascending=False)
           .astype(int)
)

semanal = semanal[semanal["rank"] <= 200]          # mesmo tamanho do chart oficial
semanal["pontos"] = 201 - semanal["rank"]

semanal["year"] = semanal["week_end"].dt.year

semanal = semanal[semanal["year"] == 2017]

cols = ["cc", "week_end", "track_id", "title", "artists", "streams", "rank", "pontos", "year"]

final = pd.concat([df[cols], semanal[cols]], ignore_index=True)

agg = (
    final.groupby(["cc", "year", "track_id"], as_index=False).agg(
        pontos=("pontos", "sum"),
        rank_medio=("rank", "mean"),
        melhor_posicao=("rank", "min"),
        semanas=("week_end", "nunique"),
        streams=("streams", "sum"),
        title=("title", "first"),
        artists=("artists", "first"),
    )
)

top40 = (
    agg.sort_values(["cc", "year", "pontos"], ascending=[True, True, False])
       .groupby(["cc", "year"])
       .head(40)
)

In [10]:
print(top40.groupby(["cc", "year"]).size().unstack())

year  2017  2018  2019  2020  2021  2022  2023  2024  2025
cc                                                        
au      40    40    40    40    40    40    40    40    40
br      40    40    40    40    40    40    40    40    40
es      40    40    40    40    40    40    40    40    40
jp      40    40    40    40    40    40    40    40    40
mx      40    40    40    40    40    40    40    40    40
us      40    40    40    40    40    40    40    40    40


In [33]:
top40 = pd.read_csv("./AnuallyCharts.csv")

In [ ]:
display(top40)

In [34]:
import re, time, json, os
import requests
import pandas as pd

# ---------- CONFIGURAÇÃO DAS APIs ----------
LRCLIB_BASE = "https://lrclib.net/api"
LYRICSOVH_BASE = "https://api.lyrics.ovh/v1"

HEADERS = {"User-Agent": "top40-letras/0.2 (seu-email-ou-repo)"}
session = requests.Session()
session.headers.update(HEADERS)

# ---------- LIMPEZA DE TÍTULO (MULTILÍNGUE) ----------
# Termos que indicam versões alternativas (acústico, ao vivo, remix, etc.)
# Inclui variações em inglês, português, espanhol e japonês.
PADROES_VERSAO = [
    # Inglês
    r"acoustic", r"live", r"remaster(?:ed)?", r"remix", r"version", r"edit",
    r"demo", r"instrumental", r"radio\s*edit", r"extended", r"deluxe",
    r"bonus", r"alternate", r"take", r"session", r"unplugged", r"stripped",
    # Português
    r"ao\s*vivo", r"acústico", r"remasterizado", r"versão", r"edição",
    r"demonstração", r"instrumental",r"Ao\s*Vivo"
    # Espanhol
    r"en\s*vivo", r"acústico", r"remasterizado", r"versión", r"edición",
    r"demostración", r"instrumental",
    # Japonês (termos comuns em títulos de faixas)
    r"アコースティック", r"ライブ", r"リマスター", r"リミックス",
    r"バージョン", r"エディット", r"デモ", r"インストゥルメンタル",
]

# Compila um único regex para remover qualquer um dos padrões acima,
# tanto entre parênteses/colchetes quanto após um hífen.
PADRAO_VERSAO_RE = re.compile(
    r"""
    \s*[\(\[]\s*(?:" + "|".join(PADROES_VERSAO) + r")[^\)\]]*\s*[\)\]]  # (Acoustic), [Live]
    |
    \s*-\s*(?:" + "|".join(PADROES_VERSAO) + r").*$                     # - Acoustic Version
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)

def limpa_titulo(t: str) -> str:
    """Remove sufixos de versão e participações (feat, ft, with, part)."""
    # 1) Remove blocos de versão (parênteses, colchetes, hífen)
    t = PADRAO_VERSAO_RE.sub("", t)
    # 2) Remove participações (feat., ft., with, part.)
    t = re.sub(r"\s*-\s*Ao Vivo.*$", "", t, flags=re.I)
    t = re.sub(r"\s*\((?:feat|ft|with|part)\.?[^)]*\)", "", t, flags=re.IGNORECASE)
    t = re.sub(r"\s*\[(?:feat|ft|with|part)\.?[^\]]*\]", "", t, flags=re.IGNORECASE)
    # 3) Remove hífens ou espaços extras no final
    t = re.sub(r"\s*-\s*$", "", t)
    return t.strip()

def primeiro_artista(a: str) -> str:
    """Extrai o primeiro artista de uma string com múltiplos artistas."""
    return re.split(r"\s*(?:\||,|&|feat\.?|ft\.?)\s*", a, flags=re.IGNORECASE)[0].strip()

# ---------- BUSCA NA LRCLIB ----------
def busca_lrclib(titulo: str, artista: str, duracao: float | None = None) -> dict | None:
    """
    Tenta correspondência exata via /api/get (com duration, se fornecida).
    Se falhar, tenta /api/search e retorna o primeiro resultado.
    """
    t, a = limpa_titulo(titulo), primeiro_artista(artista)
    params = {"track_name": t, "artist_name": a}
    if duracao is not None:
        params["duration"] = int(duracao)  # LRCLIB espera segundos inteiros

    # 1) Correspondência exata (com duração, se disponível)
    try:
        r = session.get(f"{LRCLIB_BASE}/get", params=params, timeout=15)
        if r.status_code == 200:
            return r.json()
    except requests.RequestException:
        pass

    # 2) Busca aproximada (fallback)
    try:
        r = session.get(f"{LRCLIB_BASE}/search", params={"track_name": t, "artist_name": a}, timeout=15)
        if r.status_code == 200 and r.json():
            return r.json()[0]
    except requests.RequestException:
        pass

    return None

# ---------- BUSCA NA LYRICS.OVH ----------
def busca_lyricsovh(titulo: str, artista: str) -> str | None:
    """
    Consulta a API lyrics.ovh (gratuita, sem chave).
    Retorna a letra em texto puro ou None se não encontrar.
    """
    t, a = limpa_titulo(titulo), primeiro_artista(artista)
    try:
        r = session.get(f"{LYRICSOVH_BASE}/{a}/{t}", timeout=15)
        if r.status_code == 200:
            data = r.json()
            return data.get("lyrics")  # texto puro
    except requests.RequestException:
        pass
    return None

# ---------- LOOP PRINCIPAL COM CACHE ----------
CACHE = "letras_cache.jsonl"
feitos = set()
if os.path.exists(CACHE):
    with open(CACHE, encoding="utf-8") as f:
        feitos = {json.loads(l)["track_id"] for l in f}

# Supondo que `top40` já exista no ambiente (com colunas track_id, title, artists, duration)
# Se a coluna 'duration' não existir, o código funciona sem ela (passa None).
faixas = top40[["track_id", "title", "artists"] + (["duration"] if "duration" in top40.columns else [])].drop_duplicates("track_id")

with open(CACHE, "a", encoding="utf-8") as f:
    for row in faixas.itertuples():
        if row.track_id in feitos:
            continue

        duracao = getattr(row, "duration", None)
        letra = None
        fonte = None

        # 1) Tenta LRCLIB
        res_lrclib = busca_lrclib(row.title, row.artists, duracao)
        if res_lrclib and res_lrclib.get("plainLyrics"):
            letra = res_lrclib["plainLyrics"]
            fonte = "lrclib"
            instrumental = res_lrclib.get("instrumental")
        else:
            # 2) Fallback: lyrics.ovh
            letra_ovh = busca_lyricsovh(row.title, row.artists)
            if letra_ovh:
                letra = letra_ovh
                fonte = "lyrics.ovh"
                instrumental = None
            else:
                instrumental = None

        f.write(json.dumps({
            "track_id": row.track_id,
            "encontrada": letra is not None,
            "fonte": fonte,
            "instrumental": instrumental,
            "letra": letra,
        }, ensure_ascii=False) + "\n")

        time.sleep(0.25)  # respeita as APIs

# ---------- RELATÓRIO DE ACERTOS ----------
letras = pd.read_json(CACHE, lines=True)
top40 = top40.merge(letras, on="track_id", how="left")

print("Taxa de acerto geral: {:.1%}".format(letras["encontrada"].mean()))
print("\nAcertos por fonte:")
print(letras[letras["encontrada"]]["fonte"].value_counts())
print("\nFaixas sem letra:", letras[~letras["encontrada"]].shape[0])

Taxa de acerto geral: 95.6%

Acertos por fonte:
fonte
lrclib        1253
lyrics.ovh     130
Name: count, dtype: int64

Faixas sem letra: 64


In [45]:
def saveArchiveC(data, caminho):
    import csv

    # opcional: limpar o que costuma bagunçar
    data["letra"] = (data["letra"]
                    .str.replace("\r\n", "\n", regex=False)
                    .str.strip())

    data.to_csv(
        caminho + ".csv",
        index=False,                 # evita a coluna "Unnamed: 0"
        encoding="utf-8-sig",        # o Excel abre acentos e hangul/kana corretamente
        quoting=csv.QUOTE_NONNUMERIC # texto sempre entre aspas
    )

In [ ]:
def saveArchive(data, caminho):
    import csv

    # opcional: limpar o que costuma bagunçar
    data["letra_x"] = (data["letra_x"]
                    .str.replace("\r\n", "\n", regex=False)
                    .str.strip())

    data["letra_y"] = (data["letra_y"]
                    .str.replace("\r\n", "\n", regex=False)
                    .str.strip())

    data.to_csv(
        caminho + ".csv",
        index=False,                 # evita a coluna "Unnamed: 0"
        encoding="utf-8-sig",        # o Excel abre acentos e hangul/kana corretamente
        quoting=csv.QUOTE_NONNUMERIC # texto sempre entre aspas
    )

In [35]:
letras = pd.read_json("letras_cache.jsonl", lines=True)
top40 = top40.merge(letras, on="track_id", how="left")

In [44]:
saveArchive(top40, "saidaTeste111")

In [47]:
# ============================================================
# CÉLULA ÚNICA — busca de letras (LRCLIB → lyrics.ovh → Genius)
# ============================================================
%pip install -q lyricsgenius pandas requests

import re, time, json, os
from pathlib import Path
from getpass import getpass
import requests
import pandas as pd
from lyricsgenius import Genius

# ---------- TOKEN DO GENIUS ----------
if not os.environ.get("GENIUS_ACCESS_TOKEN"):
    os.environ["GENIUS_ACCESS_TOKEN"] = getpass("CjxE91ETk4YyaUWcRcVdQkRAsvkc8Ec3qUQVTaGzoffzQ99cAVbq-e-98NATAGVL")

# ---------- CONSTANTES ----------
LRCLIB_BASE    = "https://lrclib.net/api"
LYRICSOVH_BASE = "https://api.lyrics.ovh/v1"

HEADERS = {"User-Agent": "top40-letras/0.3 (seu-email-ou-repo)"}
session = requests.Session()
session.headers.update(HEADERS)

PASTA_DADOS = Path("./dados").resolve()
PASTA_DADOS.mkdir(parents=True, exist_ok=True)
CACHE = PASTA_DADOS / "letras_cache.jsonl"
print("Cache em:", CACHE)

# ---------- HELPERS DE LIMPEZA ----------
PADROES_VERSAO = [
    # Inglês
    r"acoustic", r"live", r"remaster(?:ed)?", r"remix", r"version", r"edit",
    r"demo", r"instrumental", r"radio\s*edit", r"extended", r"deluxe",
    r"bonus", r"alternate", r"take", r"session", r"unplugged", r"stripped",
    # Português
    r"ao\s*vivo", r"acústico", r"remasterizado", r"versão", r"edição",
    r"demonstração", r"instrumental",r"Ao\s*Vivo",
    # Espanhol
    r"en\s*vivo", r"acústico", r"remasterizado", r"versión", r"edición",
    r"demostración", r"instrumental",
    # Japonês
    r"アコースティック", r"ライブ", r"リマスター", r"リミックス",
    r"バージョン", r"エディット", r"デモ", r"インストゥルメンタル",
]

PADRAO_VERSAO_RE = re.compile(
    r"""
    \s*[\(\[]\s*(?:""" + "|".join(PADROES_VERSAO) + r""")[^\)\]]*\s*[\)\]]
    |
    \s*-\s*(?:""" + "|".join(PADROES_VERSAO) + r""").*$
    """,
    flags=re.IGNORECASE | re.VERBOSE,
)

def limpa_titulo(t: str) -> str:
    t = PADRAO_VERSAO_RE.sub("", t)
    t = re.sub(r"\s*-\s*Ao Vivo.*$", "", t, flags=re.I)
    t = re.sub(r"\s*\((?:feat|ft|with|part)\.?[^)]*\)", "", t, flags=re.I)
    t = re.sub(r"\s*\[(?:feat|ft|with|part)\.?[^\]]*\]", "", t, flags=re.I)
    t = re.sub(r"\s*-\s*$", "", t)
    return t.strip()

def primeiro_artista(a: str) -> str:
    return re.split(r"\s*(?:\||,|&|feat\.?|ft\.?)\s*", a, flags=re.I)[0].strip()

# ---------- API 1: LRCLIB ----------
def busca_lrclib(titulo: str, artista: str, duracao=None) -> dict | None:
    t, a = limpa_titulo(titulo), primeiro_artista(artista)
    params = {"track_name": t, "artist_name": a}
    if duracao is not None and not pd.isna(duracao):
        params["duration"] = int(duracao)

    try:
        r = session.get(f"{LRCLIB_BASE}/get", params=params, timeout=15)
        if r.status_code == 200:
            return r.json()
    except requests.RequestException:
        pass

    try:
        r = session.get(f"{LRCLIB_BASE}/search",
                        params={"track_name": t, "artist_name": a}, timeout=15)
        if r.status_code == 200 and r.json():
            return r.json()[0]
    except requests.RequestException:
        pass
    return None

# ---------- API 2: lyrics.ovh ----------
def busca_lyricsovh(titulo: str, artista: str) -> str | None:
    t, a = limpa_titulo(titulo), primeiro_artista(artista)
    try:
        r = session.get(f"{LYRICSOVH_BASE}/{a}/{t}", timeout=15)
        if r.status_code == 200:
            return r.json().get("lyrics")
    except requests.RequestException:
        pass
    return None

# ---------- API 3: Genius ----------
genius = Genius()                    # lê GENIUS_ACCESS_TOKEN
genius.remove_section_headers = True
genius.skip_non_songs = True
genius.excluded_terms = ["(Remix)", "(Live)", "(Acoustic)", "Ao Vivo", "(Ao Vivo)"]

def busca_genius(titulo: str, artista: str) -> str | None:
    t, a = limpa_titulo(titulo), primeiro_artista(artista)
    try:
        song = genius.search_song(t, a)
        if song and song.lyrics:
            return song.lyrics
    except Exception as e:
        print(f"Erro Genius '{t}' - '{a}': {e}")
    finally:
        time.sleep(1.0)             # ~1 req/s
    return None

# ---------- BUSCA UNIFICADA (guarda todas as fontes) ----------
def busca_todas_fontes(titulo: str, artista: str, duracao=None) -> dict:
    resultado = {
        "letra_lrclib":  None,
        "letra_ovh":     None,
        "letra_genius":  None,
        "instrumental":  None,
    }

    # 1) LRCLIB
    res = busca_lrclib(titulo, artista, duracao)
    if res and res.get("plainLyrics"):
        resultado["letra_lrclib"] = res["plainLyrics"]
        resultado["instrumental"] = res.get("instrumental")

    # 2) lyrics.ovh
    letra_ovh = busca_lyricsovh(titulo, artista)
    if letra_ovh:
        resultado["letra_ovh"] = letra_ovh

    # 3) Genius (só se as duas primeiras falharam — economiza requests)
    if not resultado["letra_lrclib"] and not resultado["letra_ovh"]:
        letra_genius = busca_genius(titulo, artista)
        if letra_genius:
            resultado["letra_genius"] = letra_genius

    return resultado

# ---------- LOOP COM CACHE ----------
# Garanta que `top40` existe no kernel. Exemplo de carregamento:
# top40 = pd.read_parquet(PASTA_DADOS / "top40.parquet")
# top40 = pd.read_csv(PASTA_DADOS / "top40.csv")

feitos = set()
if CACHE.exists():
    with CACHE.open(encoding="utf-8") as f:
        feitos = {json.loads(l)["track_id"] for l in f}

cols = ["track_id", "title", "artists"]
if "duration" in top40.columns:
    cols.append("duration")
faixas = top40[cols].drop_duplicates("track_id")

with CACHE.open("a", encoding="utf-8") as f:
    for row in faixas.itertuples():
        if row.track_id in feitos:
            continue

        duracao = getattr(row, "duration", None)
        try:
            res = busca_todas_fontes(row.title, row.artists, duracao)
        except Exception as e:
            print("erro", row.track_id, e)
            continue

        res["track_id"] = row.track_id
        f.write(json.dumps(res, ensure_ascii=False) + "\n")
        time.sleep(0.25)

print("Concluído. Total de faixas no cache:", sum(1 for _ in CACHE.open(encoding="utf-8")))

# ---------- RELATÓRIO E ESCOLHA DA MELHOR FONTE ----------
letras = pd.read_json(CACHE, lines=True)

for col in ["letra_lrclib", "letra_ovh", "letra_genius"]:
    letras[f"tem_{col}"] = letras[col].notna()

print("\nCobertura por fonte:")
for col in ["letra_lrclib", "letra_ovh", "letra_genius"]:
    print(f"  {col:15s}: {letras[f'tem_{col}'].mean():.1%}")

tem_alguma = letras[["tem_letra_lrclib", "tem_letra_ovh", "tem_letra_genius"]].any(axis=1)
print(f"\nCobertura combinada: {tem_alguma.mean():.1%}")
print(f"Sem letra em nenhuma fonte: {(~tem_alguma).sum()} faixas")

def escolhe_letra(r):
    for col in ["letra_lrclib", "letra_ovh", "letra_genius"]:
        if pd.notna(r[col]):
            return r[col], col.replace("letra_", "")
    return None, None

letras[["letra", "fonte"]] = letras.apply(
    lambda r: pd.Series(escolhe_letra(r)), axis=1
)

top40_final1 = top40.merge(letras[["track_id", "letra", "fonte"]], on="track_id", how="left")
print("\nDistribuição da fonte escolhida:")
print(top40_final1["fonte"].value_counts(dropna=False))


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Cache em: /home/besome/codigo/so/dados/letras_cache.jsonl
Concluído. Total de faixas no cache: 1447

Cobertura por fonte:
  letra_lrclib   : 90.0%
  letra_ovh      : 59.9%
  letra_genius   : 4.1%

Cobertura combinada: 96.8%
Sem letra em nenhuma fonte: 47 faixas

Distribuição da fonte escolhida:
fonte
lrclib    1999
genius      60
None        53
ovh         48
Name: count, dtype: int64


In [ ]:
import osos.environ["GENIUS_ACCESS_TOKEN"] = "CjxE91ETk4YyaUWcRcVdQkRAsvkc8Ec3qUQVTaGzoffzQ99cAVbq-e-98NATAGVL"

In [54]:
saveArchiveC(top40, "saidaTeste222")

In [55]:
sem = top40[top40["fonte"].isna()]
print(len(sem), "linhas |", sem["track_id"].nunique(), "tracks únicas")

76 linhas | 64 tracks únicas


In [48]:
import re, time, json, os
import requests
import pandas as pd

BASE = "https://lrclib.net/api"
HEADERS = {"User-Agent": "top40-letras/0.1 (seu-email-ou-repo)"}
session = requests.Session()
session.headers.update(HEADERS)

def limpa_titulo(t):
    t = re.sub(r"\s*-\s*(Ao Vivo|Acústico|Remastered.*|Remix.*|.*Version.*)$", "", t, flags=re.I)
    t = re.sub(r"\s*\((feat|ft|with|part)\.?[^)]*\)", "", t, flags=re.I)
    return t.strip()

def primeiro_artista(a):
    return re.split(r"\s*(?:\||,|&)\s*", a)[0].strip()

def busca_letra(titulo, artista):
    t, a = limpa_titulo(titulo), primeiro_artista(artista)
    # 1) correspondência exata
    r = session.get(f"{BASE}/get", params={"track_name": t, "artist_name": a}, timeout=15)
    if r.status_code == 200:
        return r.json()
    # 2) busca aproximada
    r = session.get(f"{BASE}/search", params={"track_name": t, "artist_name": a}, timeout=15)
    if r.status_code == 200 and r.json():
        return r.json()[0]
    return None

# --- loop com cache ---
CACHE = "letras_cache.jsonl"
feitos = set()
if os.path.exists(CACHE):
    with open(CACHE, encoding="utf-8") as f:
        feitos = {json.loads(l)["track_id"] for l in f}

faixas = top40[["track_id", "title", "artists"]].drop_duplicates("track_id")

with open(CACHE, "a", encoding="utf-8") as f:
    for row in faixas.itertuples():
        if row.track_id in feitos:
            continue
        try:
            res = busca_letra(row.title, row.artists)
        except requests.RequestException as e:
            print("erro", row.track_id, e)
            continue
        f.write(json.dumps({
            "track_id": row.track_id,
            "encontrada": res is not None,
            "instrumental": res.get("instrumental") if res else None,
            "letra": res.get("plainLyrics") if res else None,
        }, ensure_ascii=False) + "\n")
        time.sleep(0.2)   # educado com a API

letras = pd.read_json(CACHE, lines=True)
top40 = top40.merge(letras, on="track_id", how="left")
print(letras["encontrada"].mean())   # taxa de acerto

0.9557705597788528


In [58]:
faltantes = top40[top40["encontrada"] == False]

# se quiser separar as instrumentais (não precisam de letra):
if "instrumental_x" in top40.columns:
    faltantes = top40[(top40["letra"].isna() ) & (top40["instrumental"] != True)]

faltantes = faltantes.drop_duplicates("track_id")[["track_id", "title", "artists", "cc"]]

In [52]:
faltantes.count()

Unnamed: 0        76
cc                76
year              76
track_id          76
pontos            76
rank_medio        76
melhor_posicao    76
semanas           76
streams           76
title             76
artists           76
encontrada_x      76
fonte_x            0
instrumental_x     0
letra_x            0
encontrada_y      76
fonte_y            0
instrumental_y     0
letra_y            0
encontrada        76
fonte              0
instrumental       0
letra              0
dtype: int64

In [56]:
# pip install syncedlyrics
import syncedlyrics

def busca_syncedlyrics(titulo, artista):
    t, a = limpa_titulo(titulo), primeiro_artista(artista)
    try:
        return syncedlyrics.search(f"{t} {a}", plain_only=True)  # None se não achar
    except Exception:
        return None

In [57]:
import requests

VAGALUME_KEY = "SUA_CHAVE"

def busca_vagalume(t, a):
    if not a:
        return None
    r = requests.get("https://api.vagalume.com.br/search.php",
                     params={"art": a, "mus": t, "apikey": VAGALUME_KEY}, timeout=15)
    r.raise_for_status()          # 503 vira exceção e aparece no print do loop
    j = r.json()
    if j.get("type") in ("exact", "aprox") and j.get("mus"):
        return j["mus"][0].get("text")
    return None

In [61]:
# 1. limpa sobras de merges anteriores
sobras = [c for c in top40.columns
          if c.endswith(("_x", "_y")) or c in ["Unnamed: 0", "letra_fb", "fonte_fb", "variante"]]
top40 = top40.drop(columns=sobras)

# 2. lê o cache do fallback e renomeia 'fonte' para não colidir com a 'fonte' do top40
fb = pd.read_json(CACHE, lines=True)
fb = fb.rename(columns={"fonte": "fonte_fb"})

# 3. merge só com o necessário, mantendo uma linha por track_id
fb = fb.dropna(subset=["letra_fb"]).drop_duplicates("track_id")
top40 = top40.merge(fb[["track_id", "letra_fb", "fonte_fb"]], on="track_id", how="left")

# 4. preenche letra onde estiver vazia
top40["letra"] = top40["letra"].fillna("")
top40["letra"] = top40["letra"].where(top40["letra"] != "", top40["letra_fb"]).fillna("")

In [63]:
import json, os, time

CACHE = "letras_fallback2.jsonl"
feitos = set()
if os.path.exists(CACHE):
    with open(CACHE, encoding="utf-8") as f:
        feitos = {json.loads(l)["track_id"] for l in f}

with open(CACHE, "a", encoding="utf-8") as f:
    for row in faltantes.itertuples():
        if row.track_id in feitos:
            continue
        letra, fonte = None, None
        for nome, fn in [("syncedlyrics", busca_syncedlyrics), ("vagalume", busca_vagalume)]:
            try:
                letra = fn(row.title, row.artists)
            except Exception:
                letra = None
            if letra:
                fonte = nome
                break
        f.write(json.dumps({"track_id": row.track_id, "letra_fb": letra, "fonte": fonte},
                           ensure_ascii=False) + "\n")
        time.sleep(0.5)

fb = pd.read_json(CACHE, lines=True)
dados = top40.merge(fb, on="track_id", how="left")
dados["letra"] = dados["letra"].where(dados["letra"] != None, dados["letra_fb"]).fillna("")

In [67]:
cols_letra_fb = ["letra_fb_x", "letra_fb_y", "letra_fb"]
cols_fonte_fb = ["fonte_fb", "fonte_y", "fonte"]

# primeira letra não vazia entre as três colunas do fallback
letra_fb = (dados[cols_letra_fb]
            .replace(r"^\s*$", np.nan, regex=True)
            .bfill(axis=1)
            .iloc[:, 0])

fonte_fb = dados[cols_fonte_fb].bfill(axis=1).iloc[:, 0]

# preenche só onde a letra está vazia
dados["letra"] = dados["letra"].fillna("")
vazia = dados["letra"].str.strip() == ""
achou = vazia & letra_fb.notna()

dados.loc[achou, "letra"] = letra_fb[achou]
dados.loc[achou, "fonte_x"] = fonte_fb[achou]   # mantém a origem da letra na coluna de fonte original

# remove as colunas duplicadas e renomeia a fonte original
dados = dados.drop(columns=cols_letra_fb + cols_fonte_fb)
dados = dados.rename(columns={"fonte_x": "fonte"})

sem = dados[top40["letra"].str.strip() == ""]
print(len(sem), "linhas |", sem["track_id"].nunique(), "tracks únicas sem letra")
print(top40.columns.tolist())

28 linhas | 26 tracks únicas sem letra
['cc', 'year', 'track_id', 'pontos', 'rank_medio', 'melhor_posicao', 'semanas', 'streams', 'title', 'artists', 'encontrada', 'fonte_x', 'instrumental', 'letra', 'letra_fb_x', 'fonte_fb', 'letra_fb_y', 'fonte_y']


In [68]:
dados.count()

cc                2160
year              2160
track_id          2160
pontos            2160
rank_medio        2160
melhor_posicao    2160
semanas           2160
streams           2160
title             2160
artists           2160
encontrada        2160
fonte             2084
instrumental      1867
letra             2160
Letra             2160
dtype: int64

In [ ]:
import os
import pandas as pd

COL = "letra"

faltantes = top40[top40[COL].fillna("").str.strip() == ""].drop_duplicates("track_id")
print(len(faltantes), "tracks únicas sem letra")
print(faltantes["cc"].value_counts())

CACHE = "letras_fallback2.jsonl"
if os.path.exists(CACHE) and os.path.getsize(CACHE) > 0:
    fb = pd.read_json(CACHE, lines=True)
    col_fonte = "fonte_fb" if "fonte_fb" in fb.columns else "fonte"
    print(len(fb), "linhas no cache do fallback | colunas:", fb.columns.tolist())
    print(fb[col_fonte].value_counts(dropna=False))
else:
    print("cache do fallback vazio ou inexistente")

26 tracks únicas sem letra
cc
br    13
mx     6
es     4
jp     2
au     1
Name: count, dtype: int64
64 linhas no cache do fallback | colunas: ['track_id', 'letra_fb', 'fonte']
fonte
syncedlyrics    38
NaN             26
Name: count, dtype: int64


In [72]:
import re, json, os, time

In [73]:
def primeiro_artista(a):
    return re.split(r"\s*(?:\||,)\s*", a)[0].strip()

In [74]:
def limpa_titulo(t):
    t = re.sub(r"\s*-\s*(Ao Vivo|Acústico|Remastered.*|Remix.*|.*Version.*)$", "", t, flags=re.I)
    t = re.sub(r"\s*\((feat|ft|with|part)\.?[^)]*\)", "", t, flags=re.I)
    return t.strip()

In [75]:
def variantes(titulo, artistas):
    t = limpa_titulo(titulo)
    arts = lista_artistas(artistas)
    for a in arts[:2]:
        yield "artista", t, a
    yield "titulo_original", titulo.split(" - ")[0].strip(), arts[0]
    yield "so_titulo", t, ""          # menos confiável, vai por último

In [ ]:
import re, json, os, time
import numpy as np
import pandas as pd
import requests
import syncedlyrics

# ---------- configuração ----------
COL = "letra"                       # coluna de letra em top40
CACHE = "letras_fallback3.jsonl"    # cache novo (só guarda sucessos)
RESET_CACHE = False                 # True apaga o cache e refaz tudo
PAUSA = 0.4
VAGALUME_KEY = None                 # opcional; None = não usa

session = requests.Session()
session.headers.update({"User-Agent": "top40-letras/0.1 (seu-email-ou-repo)"})

# ---------- limpeza e variantes de busca ----------
def limpa_titulo(t):
    t = re.sub(r"\s*-\s*Ao Vivo.*$", "", t, flags=re.I)                      # "- Ao Vivo", "- Ao Vivo Em ..."
    t = re.sub(r"\s*-\s*(Acústico|Remastered|Remix|.*Version).*$", "", t, flags=re.I)
    t = re.sub(r"\s*\((feat|ft|with|part|participa)[^)]*\)?", "", t, flags=re.I)
    return t.strip()

def lista_artistas(a):
    return [x.strip() for x in re.split(r"\s*\|\s*", a) if x.strip()]   # não divide em "&" nem ","

def variantes(titulo, artistas):
    t = limpa_titulo(titulo)
    arts = lista_artistas(artistas) or [""]
    for a in arts[:2]:
        yield "artista", t, a
        if "&" in a:
            yield "artista_e", t, a.replace("&", "e")   # "Matheus e Kauan"
    yield "titulo_original", titulo.split(" - ")[0].strip(), arts[0]
    yield "so_titulo", t, ""                            # menos confiável, por último

# ---------- fontes: cada uma devolve (letra, titulo_achado, artista_achado) ou None ----------
def busca_lrclib(t, a):
    r = session.get("https://lrclib.net/api/search", params={"q": f"{t} {a}".strip()}, timeout=15)
    r.raise_for_status()
    for item in r.json():
        if item.get("plainLyrics"):
            return item["plainLyrics"], item.get("trackName"), item.get("artistName")
    return None

def busca_syncedlyrics(t, a):
    letra = syncedlyrics.search(f"{t} {a}".strip(), plain_only=True)
    return (letra, None, None) if letra else None

def busca_vagalume(t, a):
    if not a or not VAGALUME_KEY:
        return None
    r = session.get("https://api.vagalume.com.br/search.php",
                    params={"art": a, "mus": t, "apikey": VAGALUME_KEY}, timeout=15)
    r.raise_for_status()
    j = r.json()
    if j.get("type") in ("exact", "aprox") and j.get("mus"):
        return j["mus"][0].get("text"), j["mus"][0].get("name"), a
    return None

fontes = [("lrclib2", busca_lrclib), ("syncedlyrics", busca_syncedlyrics)]
if VAGALUME_KEY:
    fontes.append(("vagalume", busca_vagalume))

# ---------- 1. limpeza do top40 ----------
sobras = [c for c in top40.columns
          if c.endswith(("_x", "_y")) or c in ["Unnamed: 0", "letra_fb", "fonte_fb", "variante",
                                                "match_titulo", "match_artista"]]
top40 = top40.drop(columns=sobras)
top40[COL] = top40[COL].fillna("").astype(str)
if "fonte" not in top40.columns:
    top40["fonte"] = np.nan

# ---------- 2. faltantes únicas (ignora instrumentais) ----------
vazia = top40[COL].str.strip() == ""
if "instrumental" in top40.columns:
    vazia &= top40["instrumental"] != True
faltantes = top40[vazia].drop_duplicates("track_id")
print(len(faltantes), "tracks únicas sem letra")

# ---------- 3. loop com cache (só grava sucessos) ----------
if RESET_CACHE and os.path.exists(CACHE):
    os.remove(CACHE)

feitos = set()
if os.path.exists(CACHE) and os.path.getsize(CACHE) > 0:
    with open(CACHE, encoding="utf-8") as f:
        feitos = {json.loads(l)["track_id"] for l in f}

with open(CACHE, "a", encoding="utf-8") as f:
    for row in faltantes.itertuples():
        if row.track_id in feitos:
            continue
        achou = None
        for nome, fn in fontes:
            for tipo, t, a in variantes(row.title, row.artists):
                try:
                    res = fn(t, a)
                except Exception as e:
                    print("erro", nome, row.track_id, repr(e)[:120])
                    res = None
                time.sleep(PAUSA)
                if res and res[0]:
                    achou = {"track_id": row.track_id, "letra_fb": res[0], "fonte_fb": nome,
                             "variante": tipo, "match_titulo": res[1], "match_artista": res[2]}
                    break
            if achou:
                break
        if achou:
            f.write(json.dumps(achou, ensure_ascii=False) + "\n")
            f.flush()

# ---------- 4. merge de volta em top40 (só as colunas necessárias) ----------
if os.path.exists(CACHE) and os.path.getsize(CACHE) > 0:
    fb = pd.read_json(CACHE, lines=True).drop_duplicates("track_id")
    print("achadas por fonte/variante:")
    print(fb.groupby(["fonte_fb", "variante"]).size())

    mapa_letra = fb.set_index("track_id")["letra_fb"]
    mapa_fonte = fb.set_index("track_id")["fonte_fb"]

    novas = top40["track_id"].map(mapa_letra)
    preencher = (top40[COL].str.strip() == "") & novas.notna()
    top40.loc[preencher, COL] = novas[preencher]
    top40.loc[preencher, "fonte"] = top40.loc[preencher, "track_id"].map(mapa_fonte)
else:
    fb = pd.DataFrame()
    print("o fallback não achou nada novo")

if "encontrada" in top40.columns:
    top40["encontrada"] = top40[COL].str.strip() != ""

# ---------- 5. resultado e revisão ----------
ainda = top40[top40[COL].str.strip() == ""].drop_duplicates("track_id")
print(len(ainda), "tracks únicas ainda sem letra")
print(ainda["cc"].value_counts())
display(ainda[["cc", "title", "artists"]])

# conferir se a música achada é a certa (principalmente "so_titulo")
if not fb.empty:
    rev = fb.merge(top40[["track_id", "cc", "title", "artists"]].drop_duplicates("track_id"), on="track_id")
    rev["inicio"] = rev["letra_fb"].str[:80].str.replace("\n", " / ")
    display(rev[["cc", "title", "artists", "match_titulo", "match_artista",
                 "fonte_fb", "variante", "inicio"]].sort_values("variante"))

In [86]:
import csv, os, re
import pandas as pd

_CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\u2028\u2029]")

def _limpa_texto(s, max_chars, proteger_formulas):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s)
    s = s.encode("utf-8", errors="replace").decode("utf-8")   # remove surrogates inválidos
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = _CTRL.sub("", s).strip()
    if max_chars and len(s) > max_chars:
        s = s[:max_chars]
    if proteger_formulas and s[:1] in ("=", "+", "-", "@"):
        s = " " + s
    return s

def saveArchiveB(data, caminho, sep=",", max_chars=32000,
                 proteger_formulas=True, salvar_parquet=False):
    """Salva CSV seguro para Excel/Python. Não altera o DataFrame original."""
    out = data.copy()

    # remove colunas de sobra de merges anteriores
    out = out.drop(columns=[c for c in out.columns
                            if c == "Unnamed: 0" or str(c).endswith(("_x", "_y"))])

    # limpa todas as colunas de texto (letra, title, artists, ...)
    cols_txt = out.select_dtypes(include=["object", "string"]).columns
    truncadas = 0
    for c in cols_txt:
        if c == "letra":
            truncadas = (out[c].fillna("").astype(str).str.len() > max_chars).sum()
        out[c] = out[c].map(lambda s: _limpa_texto(s, max_chars, proteger_formulas))

    arquivo = caminho if caminho.lower().endswith(".csv") else caminho + ".csv"
    os.makedirs(os.path.dirname(arquivo) or ".", exist_ok=True)

    out.to_csv(
        arquivo,
        index=False,
        sep=sep,
        encoding="utf-8-sig",            # BOM: o Excel reconhece acentos, hangul e kana
        quoting=csv.QUOTE_NONNUMERIC,    # texto sempre entre aspas
        lineterminator="\r\n",
        errors="replace",
    )

    if salvar_parquet:                    # backup sem perda (precisa de pyarrow)
        out.to_parquet(arquivo.replace(".csv", ".parquet"), index=False)

    # verificação: reler e comparar
    chk = pd.read_csv(arquivo, sep=sep, keep_default_na=False, dtype=str, encoding="utf-8-sig")
    ok_forma = chk.shape == out.shape
    ok_letra = (chk["letra"].reset_index(drop=True)
                == out["letra"].fillna("").astype(str).reset_index(drop=True)).all() \
               if "letra" in out.columns else True
    print(f"Salvo em {arquivo} | {out.shape[0]} linhas x {out.shape[1]} colunas")
    print(f"Forma preservada: {ok_forma} | Letras idênticas: {ok_letra} | Letras truncadas: {truncadas}")
    return arquivo

saveArchiveB(top40, "./completoComLetras")

Salvo em ./completoComLetras.csv | 2160 linhas x 14 colunas
Forma preservada: True | Letras idênticas: True | Letras truncadas: 0


'./completoComLetras.csv'